# Credit Risk Evaluator
This notebook compares Logistic Regression and Random Forest classifiers for predicting loan risk using 2019–2020 LendingClub lending data.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. Load Data

In [ ]:
# Load train and test data
train_df = pd.read_csv(Path('2019loans.csv'))
test_df = pd.read_csv(Path('2020Q1loans.csv'))

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape: {test_df.shape}")
train_df.head()

## 2. Preprocess Data

In [ ]:
# Convert categorical data to numeric and separate target feature for training data
X_train = train_df.drop('loan_status', axis=1)
y_train = train_df['loan_status']
X_train = pd.get_dummies(X_train)

print(f"X_train shape: {X_train.shape}")
print(f"y_train value counts:\n{y_train.value_counts()}")

In [ ]:
# Convert categorical data to numeric and separate target feature for testing data
X_test = test_df.drop('loan_status', axis=1)
y_test = test_df['loan_status']
X_test = pd.get_dummies(X_test)

print(f"X_test shape: {X_test.shape}")
print(f"y_test value counts:\n{y_test.value_counts()}")

In [ ]:
# Align train and test columns — add any missing dummy columns to the test set
missing_cols = set(X_train.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0
X_test = X_test[X_train.columns]

print(f"Aligned X_test shape: {X_test.shape}")
print(f"Missing columns added: {missing_cols}")

## 3. Baseline Models (Unscaled Data)

In [ ]:
# Train Logistic Regression on unscaled data
logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_train, y_train)
logreg_score = logreg.score(X_test, y_test)
print(f"Logistic Regression (Unscaled) Accuracy: {logreg_score:.4f}")

In [ ]:
# Train Random Forest Classifier on unscaled data
rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train, y_train)
rfc_score = rfc.score(X_test, y_test)
print(f"Random Forest (Unscaled) Accuracy: {rfc_score:.4f}")

## 4. Scale the Data

In [ ]:
# Apply StandardScaler — fit on training data only to prevent data leakage
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")

## 5. Scaled Models

In [ ]:
# Train Logistic Regression on scaled data
X_train_scaled_split, X_test_scaled_split, y_train_split, y_test_split = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42
)

lr = LogisticRegression(max_iter=5000)
lr.fit(X_train_scaled_split, y_train_split)
lr_score_scaled = lr.score(X_test_scaled_split, y_test_split)
print(f"Logistic Regression (Scaled) Accuracy: {lr_score_scaled:.4f}")

In [ ]:
# Train Random Forest Classifier on scaled data
rfc_scaled = RandomForestClassifier(n_estimators=100, random_state=42)
rfc_scaled.fit(X_train_scaled_split, y_train_split)
rfc_score_scaled = rfc_scaled.score(X_test_scaled_split, y_test_split)
print(f"Random Forest (Scaled) Accuracy: {rfc_score_scaled:.4f}")

## 6. Results Summary

In [ ]:
# Summary table of all model results
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Logistic Regression', 'Random Forest'],
    'Data': ['Unscaled', 'Unscaled', 'Scaled', 'Scaled'],
    'Accuracy': [logreg_score, rfc_score, lr_score_scaled, rfc_score_scaled]
})

results['Accuracy'] = results['Accuracy'].map('{:.2%}'.format)
print(results.to_string(index=False))

## 7. Conclusions

- **Random Forest on scaled data achieved the highest accuracy at 79.64%**, outperforming all other configurations.
- Feature scaling had a significant positive impact on Logistic Regression, improving it from 55.87% to 69.13%.
- Random Forest was more robust to unscaled data than Logistic Regression, but still benefited from scaling.
- For this dataset, ensemble methods (Random Forest) outperform linear models (Logistic Regression), likely due to non-linear relationships between features and loan status.